# 1. Data quality and preprocessing

**Question:** What did we receive, and is it suitable for analysis?

This notebook opens the public B-HOT measurements, checks their structure and
count distributions, and demonstrates the calculation shared by training and
the scoring application. One row represents one biopsy specimen.

The target is **any recorded rejection versus no rejection**. Antibody-mediated,
T cell-mediated, and mixed rejection count as positive. The source includes
transplant biopsies and native-kidney controls. The metadata do not identify
individual native-kidney controls or establish patient independence.

**Added September 27, 2026, after the presentation.** These are retrospective
data checks, not a record of EDA performed before the original model selection.

Reading order: [1. Data quality](01_data_quality_and_preprocessing.ipynb) →
[2. EDA](02_exploratory_data_analysis.ipynb) →
[3. Models and contributions](03_model_comparison_and_rna_contributions.ipynb).

## Setup

Run with this project's Python 3.12 environment (uv sync --frozen --group notebooks).
The [notebook guide](README.md) has launch and clean-checkout instructions.
Each notebook runs independently. Figures and aggregate tables go to a new
directory under results/notebooks/; complete specimen tables stay in memory.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src/kidney_biopsy").is_dir():
    raise RuntimeError("Start Jupyter from this project root or its notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from kidney_biopsy import map_diagnoses, normalize_counts, read_geo_matrix, read_rcc_archive
from kidney_biopsy.preprocessing import HOUSEKEEPING_TARGETS
from notebooks.notebook_support import (
    BK_TARGETS,
    COLORS,
    DIAGNOSIS_ORDER,
    NotebookSession,
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 4.5),
        "figure.dpi": 110,
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "axes.titlepad": 14,
    }
)
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 3)
session = NotebookSession("01_data_quality_and_preprocessing")
display(session.verify_inputs())

## 1. Load the source files and inspect their structure

The manifest above checks the physical files against the public-source record.
We take **metadata** from the GEO series matrix and **predictors** from the raw
RCC archive. The deposited processed expression matrix is not our predictor table.

The shared readers reject duplicate specimen/target names, inconsistent target
coverage, and invalid counts while loading. If that validation fails, execution
stops at the offending input; the summaries below are not a substitute for it.

In [ ]:
raw_dir = ROOT / "data/raw/rejection_public"
deposited_expression, metadata = read_geo_matrix(raw_dir / "GSE212160_series_matrix.txt.gz")
counts, batches = read_rcc_archive(raw_dir / "GSE212160_RAW.tar", specimen_ids=metadata.index)
diagnosis = metadata.histology_diagnosis
y = map_diagnoses(diagnosis)

In [ ]:
overview = pd.DataFrame(
    {
        "table": [
            "Raw counts",
            "Deposited processed expression (unused)",
            "Metadata",
            "Assay metadata",
        ],
        "rows": [len(counts), len(deposited_expression), len(metadata), len(batches)],
        "columns": [
            counts.shape[1],
            deposited_expression.shape[1],
            metadata.shape[1],
            batches.shape[1],
        ],
    }
)
display(overview)
display(counts.iloc[:3, :6].set_axis(["Specimen A", "Specimen B", "Specimen C"]))
display(counts.dtypes.astype(str).value_counts().rename("number_of_targets").to_frame())
session.table("table_inventory", overview)

## 2. Check missingness, joins, identifiers, labels, and count validity

The source field is misspelled 'histology diganosis of rejection'; the reader
retains it and exposes the alias 'histology_diagnosis'. Unknown labels are errors.
Zero counts are valid measurements and are counted separately from missing values.

In [ ]:
values = counts.to_numpy()
audit = pd.DataFrame(
    [
        ("Missing count values", int(counts.isna().sum().sum()), "Stop if present"),
        (
            "Nonnumeric count columns",
            int(sum(not pd.api.types.is_numeric_dtype(t) for t in counts.dtypes)),
            "Stop if present",
        ),
        ("Non-finite count values", int((~np.isfinite(values)).sum()), "Stop if present"),
        ("Negative counts", int((values < 0).sum()), "Stop if present"),
        ("Noninteger counts", int((values != np.floor(values)).sum()), "Inspect if present"),
        ("Duplicate specimen IDs", int(counts.index.duplicated().sum()), "Stop if present"),
        ("Duplicate target names", int(counts.columns.duplicated().sum()), "Stop if present"),
        (
            "Unmatched metadata/count IDs",
            len(set(metadata.index) ^ set(counts.index)),
            "Stop if present",
        ),
        (
            "Missing housekeeping targets",
            len(set(HOUSEKEEPING_TARGETS) - set(counts.columns)),
            "Stop if present",
        ),
        (
            "Unknown or missing diagnoses",
            int((~diagnosis.isin(DIAGNOSIS_ORDER)).sum()),
            "Stop if present",
        ),
        ("All-zero specimens", int(counts.eq(0).all(axis=1).sum()), "Inspect assay quality"),
        (
            "Duplicate raw profiles (all members)",
            int(counts.duplicated(keep=False).sum()),
            "Review repeated measurements",
        ),
    ],
    columns=["check", "observed", "action"],
)
display(audit)
completeness = pd.DataFrame(
    {
        "dtype": metadata.dtypes.astype(str),
        "missing": metadata.isna().sum(),
        "blank": metadata.fillna("").eq("").sum(),
        "distinct_values": metadata.nunique(),
    }
)
display(completeness)
session.table("data_quality_checks", audit)
session.table("metadata_completeness", completeness)
display(
    Markdown(
        f"**Observed:** {len(counts):,} specimens and {counts.shape[1]} targets loaded. "
        f"The count table has {int(counts.isna().sum().sum()):,} missing values and "
        f"{int(counts.duplicated(keep=False).sum())} specimens in duplicate raw profiles. "
        "No specimen is removed by this notebook."
    )
)

## 3. Are counts unusually sparse or extreme?

Total signal and the proportion of zeros can reveal specimens worth inspecting.
Here, an outlier means outside the usual 1.5 × IQR fences on **log total count**.
That is a descriptive rule, not a laboratory acceptance threshold. The assay
reader uses the 770 endogenous/housekeeping targets; this is not a complete
NanoString instrument-quality assessment of control probes and imaging.

In [ ]:
specimen_qc = pd.DataFrame(
    {
        "total_count": counts.sum(axis=1),
        "zero_fraction": counts.eq(0).mean(axis=1),
        "housekeeping_log2_mean": np.log2(counts[list(HOUSEKEEPING_TARGETS)] + 1).mean(axis=1),
    }
)
log_total = np.log10(specimen_qc.total_count + 1)
q1, q3 = log_total.quantile([0.25, 0.75])
fences = (q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))
outside = (log_total < fences[0]) | (log_total > fences[1])
qc_summary = specimen_qc.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
display(qc_summary)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
for ax, series, label in zip(
    axes,
    [log_total, specimen_qc.zero_fraction, specimen_qc.housekeeping_log2_mean],
    ["log10(total count + 1)", "Fraction of zero targets", "Mean log2(housekeeping count + 1)"],
):
    ax.hist(series, bins=35, color=COLORS[0], edgecolor="white")
    ax.set(xlabel=label, ylabel="Specimens")
fig.suptitle("Raw-count checks across all 1,395 specimens")
session.figure("raw_count_checks", fig)
plt.show()
session.table("specimen_qc_summary", qc_summary)
display(
    Markdown(
        f"**Observed:** {int(outside.sum())} specimens fall outside the log-total-count fences. "
        f"The median zero fraction is {specimen_qc.zero_fraction.median():.1%}. "
        "These summaries flag observations for review; they do not establish poor assay quality."
    )
)

### Follow up on the unusual totals

Count how the flagged specimens are distributed across cohorts and diagnoses,
then compare their housekeeping signal and sparsity with the remaining specimens.
Higher or lower total counts can have several explanations. These measurements
alone cannot distinguish tissue composition, RNA quantity, and assay conditions.

In [ ]:
qc_review = specimen_qc.assign(
    cohort=metadata.cohort,
    diagnosis=diagnosis,
    below_fence=log_total.lt(fences[0]),
    above_fence=log_total.gt(fences[1]),
    unusual_total=outside,
)
outlier_groups = qc_review.groupby(["cohort", "diagnosis"]).agg(
    specimens=("unusual_total", "size"),
    below_fence=("below_fence", "sum"),
    above_fence=("above_fence", "sum"),
)
outlier_comparison = (
    qc_review.groupby("unusual_total")[["total_count", "zero_fraction", "housekeeping_log2_mean"]]
    .median()
    .rename(index={False: "Within fences", True: "Outside fences"})
)
display(outlier_groups)
display(outlier_comparison)
session.table("unusual_totals_by_group", outlier_groups)
session.table("unusual_totals_medians", outlier_comparison)
display(
    Markdown(
        f"**Observed:** {int(qc_review.below_fence.sum())} specimens have unusually low "
        f"totals and {int(qc_review.above_fence.sum())} have unusually high totals. "
        "They passed the structural count checks. The tables show which groups "
        "contribute to these tails without assigning a laboratory cause."
    )
)

## 4. Inspect the 12 housekeeping targets

Every specimen uses all 12 named targets for normalization. We inspect both their
distributions and any zeros. Different target abundances are expected; a single
high or low count does not by itself establish a failed specimen.

In [ ]:
hk_log = np.log2(counts[list(HOUSEKEEPING_TARGETS)] + 1)
hk_summary = pd.DataFrame(
    {
        "raw_min": counts[list(HOUSEKEEPING_TARGETS)].min(),
        "raw_median": counts[list(HOUSEKEEPING_TARGETS)].median(),
        "raw_max": counts[list(HOUSEKEEPING_TARGETS)].max(),
        "zero_count": counts[list(HOUSEKEEPING_TARGETS)].eq(0).sum(),
    }
)
display(hk_summary)
fig, ax = plt.subplots(figsize=(11, 4.5), layout="constrained")
ax.boxplot(hk_log.to_numpy(), tick_labels=hk_log.columns, showfliers=False)
ax.tick_params(axis="x", rotation=45)
ax.set(ylabel="log2(raw count + 1)", title="Housekeeping targets: median and spread")
ax.text(
    0,
    -0.32,
    "Outlier markers omitted for readability; all specimens enter the summaries.",
    transform=ax.transAxes,
    fontsize=9,
)
session.figure("housekeeping_distributions", fig)
plt.show()
session.table("housekeeping_summary", hk_summary)

## 5. Work through the shared normalization

For each specimen, subtract its mean housekeeping log count from every target's
log count:

**normalized target = log2(raw count + 1) − mean(log2(12 housekeeping counts + 1))**

This calculation has no fitted cross-specimen parameters. It removes the 12
housekeeping targets from predictors, leaving 758 assay targets. Any later
scaling or feature selection must be fitted using training specimens only.

In [ ]:
X = normalize_counts(counts)
example_id = counts.index[0]  # Deterministic illustration, not a selected clinical case.
targets = ["IFNG", "GBP4", *BK_TARGETS]
worked = pd.DataFrame(
    {
        "raw_count": counts.loc[example_id, targets],
        "log2_count_plus_one": np.log2(counts.loc[example_id, targets] + 1),
        "housekeeping_mean": hk_log.loc[example_id].mean(),
        "shared_normalized_value": X.loc[example_id, targets],
    }
)
manual = np.log2(counts.loc[example_id] + 1) - hk_log.loc[example_id].mean()
np.testing.assert_allclose(manual.loc[X.columns], X.loc[example_id], rtol=0, atol=1e-12)
assert set(X.columns).isdisjoint(HOUSEKEEPING_TARGETS)
assert set(X.columns).isdisjoint(metadata.columns)
display(worked)
print(f"Shared and manual calculation agree. Predictor matrix: {X.shape}.")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].hist(
    np.log2(counts.drop(columns=list(HOUSEKEEPING_TARGETS)) + 1).to_numpy().ravel(),
    bins=70,
    color=COLORS[0],
)
axes[1].hist(X.to_numpy().ravel(), bins=70, color=COLORS[1])
axes[0].set(
    title="Before housekeeping adjustment",
    xlabel="log2(raw count + 1)",
    ylabel="Target measurements",
)
axes[1].set(
    title="After housekeeping adjustment",
    xlabel="Normalized assay value",
    ylabel="Target measurements",
)
session.figure("normalization_before_after", fig)
plt.show()

## 6. What do these checks establish?

Passing the structural checks supports using the recorded measurements in the
defined workflow. It does not establish assay compatibility for arbitrary
uploads or supply missing patient, infection, or native-kidney identifiers.
The next notebook examines diagnosis composition, correlated targets, and batches.

Source context: [project data specification](../docs/PROJECT_SPEC.md),
[public-source manifest](../data/manifest.json), and
[study composition review](../docs/references/STUDY_AUDIT_20260919.md).

In [ ]:
session.finish(
    {
        "specimens": len(counts),
        "raw_targets": counts.shape[1],
        "predictors": X.shape[1],
        "missing_counts": int(counts.isna().sum().sum()),
        "duplicate_raw_profiles": int(counts.duplicated(keep=False).sum()),
        "duplicate_normalized_profiles": int(X.duplicated(keep=False).sum()),
        "log_total_count_outliers": int(outside.sum()),
        "housekeeping_zeros": int(hk_summary.zero_count.sum()),
        "excluded_specimens": 0,
    },
    {"outlier_rule": "log10(total count + 1), 1.5 IQR fences; descriptive only"},
)